# Tutorial: from "I have a chemistry state" to "I have a certified bias bar"

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kootru-repo/cumulant-residual-cert/blob/main/notebooks/00_tutorial.ipynb)


This is the front door for users who have just installed the library. By the end you will:

1. Understand the one inequality the whole library is about.
2. Know which of four paths fits your situation.
3. See **the same physical state certified through all four paths** so you can compare what each one buys you.
4. Apply a go / no-go decision rule against your own tolerance.

If you already know which path you want, skip to `05_cookbook.ipynb` for direct recipes. If you want the theorem mechanics first, see `01_quickstart.ipynb`.

## The inequality

Let $W$ be a charge-neutral fermionic word of length $\le r$ on a $U(1)$-invariant state $\rho$ (i.e. $[\rho, \widehat N] = 0$). The cumulant-truncated estimate $\langle W \rangle_{\mathrm{trunc}}$ reconstructed from the 1- and 2-RDM has a deterministic bias bound

$$
\bigl| \langle W \rangle_{\rho} - \langle W \rangle_{\mathrm{trunc}, \rho} \bigr|
\;\le\;
C_W \cdot \Delta_{r, U(1)}^{\mathrm{cat}}(\rho),
$$

where $C_W$ is an integer-valued partition-lattice constant (depends only on the word's charge structure) and $\Delta$ is the high-cumulant envelope of the state restricted to the catalog. The whole library is plumbing for this one inequality.

What is supplied per path differs only in *how $\Delta$ is obtained*.

## The decision tree

Ask these four questions in order:

```
Q1. Is the prepared state a Slater determinant (HF / DFT mean-field)
    in the canonical molecular orbital basis?
      yes -> Path 1 (Bernoulli zero-cost):    Delta = 0 exactly. Done.
      no  -> next question.

Q2. Do you have 1-, 2-, 3-, 4-RDMs of your state
    (e.g. produced by a PySCF CISD / CASCI / FCI run)?
      yes -> Path 2 (from RDMs):              Delta evaluated exactly via Mobius.
      no  -> next question.

Q3. Do you have shadow records on your state?
      yes, random-Pauli       -> Path 3 (delta_ucb).
      yes, matchgate/Gaussian -> Path 4 (delta_ucb_from_majorana_moments).
      no                      -> next question.

Q4. None of the above?
      Compute Delta by hand from state knowledge (e.g. an analytic
      cumulant bound) and pass it via certify(catalog, delta=<value>),
      or escalate to direct measurement of the observable.
```

The rest of this notebook walks each path on **the same physical state** so the differences are visible side by side.

In [ ]:
# Bootstrap on Colab / fresh env (skipped if the library is already importable).
# Local users running after 'uv sync --extra dev' skip the install.
# On Colab the install uses uv pip install --system, which is uv's native
# compatibility subcommand for the system Python environment (not the pip tool).
import importlib.util as _ilu
if _ilu.find_spec("cumulant_residual_cert") is None:
    import subprocess as _sp, os as _os
    # Pin uv to a known directory so we invoke it by absolute path. UV_INSTALL_DIR
    # is the installer's explicit override; without it the location depends on
    # CARGO_HOME / XDG_BIN_HOME inheritance (Colab differs from local Linux).
    _UV_DIR = "/tmp/uv-bootstrap"
    _os.makedirs(_UV_DIR, exist_ok=True)
    _env = {**_os.environ, "UV_INSTALL_DIR": _UV_DIR}
    _sp.check_call(
        "curl -LsSf https://astral.sh/uv/install.sh | sh",
        shell=True, executable="/bin/bash", env=_env,
    )
    _UV = f"{_UV_DIR}/uv"
    _sp.check_call([
        _UV, "pip", "install", "--system", "-q",
        "cumulant_residual_cert@git+https://github.com/kootru-repo/cumulant-residual-cert.git",
        "numpy",
    ])

import cumulant_residual_cert
import numpy as np
print(f"cumulant_residual_cert {cumulant_residual_cert.__version__} ready")

## A single state to certify, walked through every path

We use a small two-electron superposition on $n = 4$ spin-orbitals:
$$
|\psi\rangle = \cos\theta \, |1100\rangle + \sin\theta \, |0011\rangle,
\qquad
\rho = |\psi\rangle \langle \psi |.
$$
Both basis kets have $N = 2$, so $\rho$ is $U(1)$-invariant. For $\theta = \pi/5$ it is a non-trivial correlated state (Paths 2, 3, 4 will see a positive $\Delta$). For $\theta = 0$ it collapses to the Hartree-Fock determinant $|1100\rangle$ (Path 1 sees $\Delta = 0$ exactly).

We target the worst-case catalog word $W = n_1 n_2 n_3 n_4$ on sites $(1, 2, 3, 4)$, which has block-refined constant $\widehat B^{\mathrm{charge}}_4(W) = 5$ on the chemistry catalog.

In [ ]:
import numpy as np

from cumulant_residual_cert import Catalog, certify, constants
from cumulant_residual_cert._fermion import letter_op
from itertools import product as iproduct

n = 4
theta = np.pi / 5
psi = np.zeros(2 ** n, dtype=complex)
psi[int('1100', 2)] = np.cos(theta)
psi[int('0011', 2)] = np.sin(theta)
rho = np.outer(psi, psi.conj())

catalog = Catalog.chemistry_r4()
target_word = next(w for w in catalog if w.name == 'n_i n_j n_k n_ell')
sites = (1, 2, 3, 4)
sites_per_word = [
    (1, 2, 3), (1, 2, 3),                         # length-3 words
    (1, 2, 3, 4), (1, 2, 3, 4), (1, 2, 3, 4),     # length-4 words
]
print(f'State: 2-electron superposition at theta = pi/5 = {theta:.4f} rad')
print(f'Target word: {target_word.name}')
print(f'Block-refined constant C_W = {constants.block_refined(catalog.r, target_word)}')

## Path 1 — Hartree-Fock baseline (zero cost)

**Situation.** Your state is a Slater determinant in the canonical MO basis. The worked-example theorem proves $\Delta = 0$ identically on every occupation-basis diagonal product state, which is the class HF determinants fall into when the dictionary basis matches their canonical orbitals.

**What you do.** Pass the converged mean-field object to `adapters.pyscf.from_mean_field(..., user_asserts_bernoulli_class=True)`. In this self-contained notebook we simulate that step by constructing the Slater state $|1100\rangle$ directly and feeding $\Delta = 0$ to `certify` with the right provenance label.

**Expected.** Every bound is exactly zero. No measurement, no shadows, no RDMs needed.

In [ ]:
# The HF Slater state and a hand-computed certificate. With PySCF
# installed you would replace this with:
#   from pyscf import gto, scf
#   from cumulant_residual_cert.adapters.pyscf import from_mean_field
#   mol = gto.M(atom='H 0 0 0; H 0 0 0.74', basis='sto-3g')
#   mf = scf.RHF(mol).run()
#   est = from_mean_field(mf, catalog, user_asserts_bernoulli_class=True)
result_path1 = certify(
    catalog,
    delta=0.0,
    level='block_refined',
    delta_provenance='closed_form_bernoulli',
)
print('Path 1 (HF baseline):')
print(f'  Delta              = {result_path1.delta}')
print(f'  delta_provenance   = {result_path1.delta_provenance}')
print(f'  bound on target W  = {result_path1.bounds[target_word.name]}')
print(f'  bounds on every catalog word = {set(result_path1.bounds.values())}    (expected {{0}})')

**When this applies and when it does not.** Path 1 applies when (a) your prepared state is a Slater determinant or a number-conserving free Gibbs state, and (b) the dictionary basis you certify against is the canonical orbital basis of that determinant. If either of those fails (correlated state; rotated basis), move to Path 2.

## Path 2 — Post-Hartree-Fock from RDMs (exact $\Delta$)

**Situation.** Your state has correlation: it is a CISD / CASCI / NEVPT2 / FCI solution (or any other non-Slater $U(1)$-invariant state). You have its 1-, 2-, 3-, 4-RDMs as numpy arrays (most chemistry codes that produce CI-like wavefunctions also produce RDMs as a byproduct).

**What you do.** Pass the RDMs to `adapters.pyscf.from_rdms`. The Mobius formula and the in-house normal-ordering routine evaluate every catalog cumulant exactly from those tensors. No additional measurement required.

**Expected.** Positive $\Delta$ (the state has nonzero higher-order cumulants), exact value (no Monte Carlo, no UCB).

In [ ]:
from cumulant_residual_cert.adapters.pyscf import from_rdms

def build_rdm(rho, k, n):
    shape = (n,) * (2 * k)
    rdm = np.zeros(shape, dtype=complex)
    for idx in iproduct(range(n), repeat=2 * k):
        p, q = idx[:k], idx[k:]
        op = letter_op('I', 1, n)
        for s0 in p: op = op @ letter_op('a_dag', s0 + 1, n)
        for s0 in reversed(q): op = op @ letter_op('a', s0 + 1, n)
        rdm[idx] = np.trace(rho @ op)
    return rdm

rdm1, rdm2, rdm3, rdm4 = (build_rdm(rho, k, n) for k in (1, 2, 3, 4))

est_path2 = from_rdms(
    rdm1=rdm1, rdm2=rdm2, rdm3=rdm3, rdm4=rdm4,
    catalog=catalog, sites_per_word=sites_per_word,
    level='block_refined',
)
print('Path 2 (post-HF from RDMs):')
print(f'  Delta              = {est_path2.delta:.6f}    (exact, U(1)-invariant)')
print(f'  delta_is_exact     = {est_path2.delta_is_exact}')
print(f'  delta_provenance   = {est_path2.bound.delta_provenance}')
print(f'  bound on target W  = {est_path2.bound.bounds[target_word.name]:.6f}')

**When this applies and when it does not.** Path 2 applies when you have RDMs up to length $r$ (so length-3 catalog words need at least 3-RDM, length-4 need 4-RDM). When you only have 1- and 2-RDM and the catalog contains length-$\ge 3$ words, route the missing length-$\ge 3$ subwords through Path 3 or 4 (shadow estimation).

## Path 3 — Random-Pauli shadow data

**Situation.** You measured random-Pauli shadows on your state (e.g. via Qiskit / OpenFermion / a custom protocol). You have a list of `(basis, outcomes)` shadow shots.

**What you do.** Pass the shots to `delta_ucb`. The library Bonferroni-corrects over every Pauli string in the catalog's subword expansions and propagates Hoeffding radii through the Mobius transform, returning a one-sided UCB on $\Delta$ at the requested confidence.

**Expected.** Positive $\Delta$ upper bound (looser than Path 2 because of finite-shot uncertainty), holds with probability $\ge 1 - \alpha$ simultaneously over the catalog.

**Caveat.** The built-in expansion is dense and refuses to run past $n_{\mathrm{qubits}} = 10$. For chemistry-scale registers, supply per-Pauli estimates directly via `delta_ucb_from_subword_moments`.

In [ ]:
from cumulant_residual_cert import delta_ucb
from cumulant_residual_cert.diagnostic import collect_shadows  # local-simulator helper

shadows = collect_shadows(rho, n=n, M=1000, seed=42)
ucb_path3 = delta_ucb(
    shadow_samples=shadows,
    catalog=catalog,
    sites_per_word=sites_per_word,
    n_qubits=n,
    confidence=0.95,
)
result_path3 = certify(
    catalog, delta=ucb_path3.delta_ucb,
    level='block_refined',
    delta_provenance='ucb_random_pauli',
)
print('Path 3 (random-Pauli shadows):')
print(f'  Delta UCB (95%)    = {ucb_path3.delta_ucb:.6f}    (statistical upper bound, 1000 shots)')
print(f'  Pauli strings in Bonferroni union = {ucb_path3.n_paulis}')
print(f'  delta_provenance   = {result_path3.delta_provenance}')
print(f'  bound on target W  = {result_path3.bounds[target_word.name]:.6f}')

## Path 4 — Matchgate / fermionic-Gaussian shadow data

**Situation.** You measured matchgate shadows (Wan-Hadfield-Cleve-Babbush 2022, Zhao-Rubin-Miyake-Babbush 2021). The output is empirical $(\hat{\langle \gamma_S \rangle}, \mathrm{rad}_S)$ for every degree-$2k$ Majorana product $\gamma_S$ in the catalog's decomposition. This protocol avoids the $3^{|P|}$ random-Pauli Jordan-Wigner penalty.

**What you do.** Pass the per-Majorana-product `(mean, radius)` dict to `delta_ucb_from_majorana_moments`. The library handles the dictionary-letter to Majorana decomposition and the Mobius assembly.

**Status.** The wrapper is functional; a built-in matchgate snapshot estimator (Pfaffian + orthogonal-matrix algebra) is planned for a later release. Until then the caller supplies the Majorana moments.

**For the tutorial.** We compute the exact Majorana moments analytically from our dense state so the path is reproducible without external matchgate-shadow code.

In [ ]:
from cumulant_residual_cert import delta_ucb_from_majorana_moments
from cumulant_residual_cert._fermion import _kron

# Build dense Majorana operators. Site p (1-based) has Majoranas
# gamma_{2p-1}, gamma_{2p}. In JW form: gamma_{2p-1} = Z .. Z X I .. I,
# gamma_{2p} = Z .. Z Y I .. I.
I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

def majorana_dense(j, n):
    site = (j + 1) // 2
    is_odd = (j % 2 == 1)
    blocks = []
    for s in range(1, n + 1):
        if s < site: blocks.append(Z)
        elif s == site: blocks.append(X if is_odd else Y)
        else: blocks.append(I2)
    return _kron(blocks)

def maj_product_dense(indices, n):
    op = np.eye(2 ** n, dtype=complex)
    for j in indices: op = op @ majorana_dense(j, n)
    return op

# Enumerate Majorana products that show up in the catalog's letter-to-Majorana
# decomposition and compute each one's exact expectation. In a real matchgate-
# shadow workflow these would be empirical means with non-zero radii.
from cumulant_residual_cert._majorana import word_majorana_decomposition

needed_indices = set()
for w, ws_sites in zip(catalog, sites_per_word):
    from itertools import combinations
    m = w.length
    for k in range(1, m + 1):
        for B in combinations(range(1, m + 1), k):
            key = tuple(sorted(B))
            sub_letters = tuple(w.letters[i - 1] for i in key)
            sub_sites = tuple(ws_sites[i - 1] for i in key)
            decomp = word_majorana_decomposition(sub_letters, sub_sites)
            needed_indices.update(decomp.keys())

tiny_radius = 0.0001     # near-exact estimates; in real workflows you'd derive from shot count
majorana_moments = {}
for indices in needed_indices:
    if not indices:
        majorana_moments[indices] = (1.0 + 0j, 0.0)
        continue
    op = maj_product_dense(indices, n)
    mean = complex(np.trace(rho @ op))
    majorana_moments[indices] = (mean, tiny_radius)

ucb_path4 = delta_ucb_from_majorana_moments(
    majorana_moments=majorana_moments,
    catalog=catalog,
    sites_per_word=sites_per_word,
    confidence=0.95,
    n_protocol_terms=len(majorana_moments),
)
result_path4 = certify(
    catalog, delta=ucb_path4.delta_ucb,
    level='block_refined',
    delta_provenance='ucb_matchgate_shadows',
)
print('Path 4 (matchgate / fermionic-Gaussian shadows):')
print(f'  Delta UCB          = {ucb_path4.delta_ucb:.6f}    (tight: tiny per-Majorana radii)')
print(f'  Majorana products in Bonferroni union = {ucb_path4.n_paulis}')
print(f'  delta_provenance   = {result_path4.delta_provenance}')
print(f'  bound on target W  = {result_path4.bounds[target_word.name]:.6f}')

## Side by side: what each path produces

All four paths produce a valid certified bound on the same target word. They differ in how $\Delta$ is obtained, which in turn affects the tightness.

In [ ]:
rows = [
    ('Path 1', 'HF Slater state (worked-example theorem)',  0.0, result_path1.bounds[target_word.name],  result_path1.delta_provenance),
    ('Path 2', 'Same correlated state from RDMs',           est_path2.delta, est_path2.bound.bounds[target_word.name], est_path2.bound.delta_provenance),
    ('Path 3', 'Random-Pauli shadows (1000 shots)',         ucb_path3.delta_ucb, result_path3.bounds[target_word.name],  result_path3.delta_provenance),
    ('Path 4', 'Matchgate-shadow moments (tiny radii)',     ucb_path4.delta_ucb, result_path4.bounds[target_word.name],  result_path4.delta_provenance),
]
print(f'{"path":6s} {"setup":48s} {"Delta":>12s} {"bound on W":>14s}  provenance')
print('-' * 110)
for path, setup, delta, bound, prov in rows:
    print(f'{path:6s} {setup:48s} {delta:>12.6f} {bound:>14.6f}  {prov}')

**Reading this table.**

- **Path 1** gives zero exactly, but only because we swapped the correlated state for the HF Slater. The Bernoulli-class theorem does not apply to the correlated state of this notebook; that is what the other three paths handle.
- **Path 2** is the exact $\Delta$ for the correlated state, using only 1-4 RDMs. This is the gold-standard route when RDMs are available.
- **Path 3** is the statistical upper bound. For 1000 shots, the bound is looser than Path 2 because of the $3^{|P|}$ random-Pauli range factor.
- **Path 4** with tiny synthetic radii is essentially as tight as Path 2; a real matchgate-shadow experiment will give a finite radius determined by shot count and Pauli structure, but without the $3^{|P|}$ penalty.

In a real workflow you choose **one** path that matches your data, not all four.

## The go / no-go decision rule

You picked a target tolerance $\epsilon$ for the truncation bias (e.g. $\epsilon = 5 \times 10^{-3}$). The decision rule is:

$$
\text{Bound on } W \le \epsilon \;\Longrightarrow\; \text{truncated estimate certified at tolerance } \epsilon.
$$

If the bound exceeds your tolerance, your options are:

1. **Tighten $\Delta$.** Take more shadow shots (Paths 3 and 4) or supply higher-order RDMs (Path 2).
2. **Loosen $\epsilon$** if scientifically justified for the downstream consumer of the estimate.
3. **Measure $W$ directly** instead of truncating: when the bias bar is too loose, the certificate is doing its job by telling you the closure is not certified on this state.

In [ ]:
epsilon = 0.05      # your tolerance; pick what your downstream needs
print(f'Tolerance epsilon = {epsilon}')
for path, setup, delta, bound, prov in rows:
    verdict = 'ACCEPT' if bound <= epsilon else 'REJECT (escalate)'
    print(f'  {path}: bound = {bound:.6f}  ->  {verdict}')

## What comes next

- `01_quickstart.ipynb`: theorem-mechanics walkthrough with **expected vs actual** at every step (read this if you want the theorem to feel concrete before you trust it).
- `02_bernoulli_worked.ipynb`: random Bernoulli-class state at $n = 6$, direct connected-cumulant evaluation, $\Delta = 0$ confirmed at machine precision.
- PySCF mean-field workflow lives in `05_cookbook.ipynb` Recipe 1 (HF / DFT baseline on H$_2$ STO-3G).
- `04_real_shadow_data.ipynb`: pluggable shadow-data routing with simulator + IBM / Rigetti / IonQ / IQM stubs.
- `05_cookbook.ipynb`: nine direct recipes (custom catalog, JSON persistence, OpenFermion / Qiskit-Nature operator conversion, ...).

Decision tree → pick a path → cookbook recipe → save the certificate. That is the full user journey.